In [ ]:
%pip install -r requirements.txt

In [ ]:
import random
import math
import kdtree
from statistics import mean
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.patches import Polygon as MPolygon, Circle
from map_generators.radnom_walk import generate_random_walk_map
from map_generators.non_intersecting_polygons import generate_non_intersecting_polygons
from map_generators.rect_with_gap import generate_rects_with_gap
from tests.read_test_map import read_test_map
from visualization.map_visualization import show_map_vectorized
from visualization.rrt_visualization import visualize_rrt
from tests.check_results import construct_path_from_node
from utils.node import RRTNode
from review_checks import review_checks
from shapely.geometry import Polygon as ShapelyPolygon, Point, LineString
from typing import List, Tuple, Optional

State = Tuple[float, float]
Obstacles = List[List[State]]
SearchResult = Tuple[bool, Optional[RRTNode], int, kdtree.KDNode,
                     List[Tuple[RRTNode, int]]]

![image.png](./media/tree.png)

# Алгоритм RRT на основе сэмплирования для задачи поиска пути
В данной лабораторной работе мы изучим задачу поиска пути в непрерывном пространстве для агента-робота, используя подход на основе сэмплирования. Основной метод, который мы будем рассматривать, — это алгоритм Rapidly-exploring Random Tree (RRT), широко применяемый для планирования траекторий в сложных конфигурационных пространствах.

Алгоритм RRT опирается на построение специальной структуры данных — дерева, которое быстро охватывает область поиска за счет последовательного добавления новых узлов. Каждый узел дерева, за исключением корня, имеет ровно одного родителя. На каждом шаге алгоритм генерирует новый узел и пытается присоединить его к существующему дереву. Конечная цель заключается в том, чтобы достичь области, близкой к целевой точке, посредством последовательного расширения дерева.

Процесс расширения дерева RRT можно описать следующим образом:

1) **Генерация случайной точки:** выбрать случайную точку x<sub>rand</sub> пространстве.  
2) **Поиск ближайшего узла:** найти узел x<sub>near</sub> в текущем дереве, ближайший к точке x<sub>rand</sub> 
3) **Расширение дерева:** создать новый узел x<sub>new</sub> в направлении от точки x<sub>near</sub> к x<sub>rand</sub>.  

Преимущество алгоритмов RRT и других методов на основе сэмплинга заключается в их применимости для задач поиска пути в непрерывных пространствах с учетом различных ограничений — геометрических (например, форма и размеры агента) и кинематических (ограничения на скорость и ускорение). Эти методы особенно эффективны при решении задач высокой размерности, таких как планирование движений манипуляторов.

Однако в данной лабораторной работе мы сосредоточимся на ключевых идеях алгоритмов на основе сэмплинга, применяя их к задаче поиска пути на двумерной плоскости.

# Представление пространства

Для моделирования задачи поиска пути и построения дерева RRT в этой лабораторной работе используется двумерное пространство, в котором могут находиться препятствия, заданные в виде многоугольников.

## Описание препятствий

Препятствия определяются как наборы вершин многоугольников. Каждое препятствие представлено списком координат вершин в формате *(x, y)*. Например, список `obstacles_sample` содержит два препятствия:

In [ ]:
obstacles_sample = [[(2, 2), (3, 3), (2, 4), (1, 3)],  # ромб
                    [(5, 5), (6, 7), (7, 6)],  # треугольник
                   ]

show_map_vectorized(10, 10, obstacles_sample, show_cords=False, show_obstacles_index=True, show_vertexes=True, label="Карта с препятствиями")

## Узлы дерева и поиск ближайших соседей
Алгоритм RRT строит дерево поиска, сэмплируя новые состояния и пытаясь соединить их с уже существующими узлами дерева. Одной из ключевых операций в процессе работы алгоритма является поиск ближайшего узла дерева к новому, только что сэмплированному состоянию. Эта операция будет выполняться многократно и желательно реализовать её эффективно.

Для реализации данной задачи мы будем использовать внешний модуль `kdtree`, который предоставляет структуру данных — [k-d-дерево](https://ru.wikipedia.org/wiki/K-d-дерево). K-d-дерево позволяет эффективно выполнять операции добавления элементов и поиска ближайших соседей с хорошей асимптотикой:

* Добавление элемента: **O(h)**.
* Поиск ближайшего соседа: в среднем **O(log n)** для сбалансированного дерева
  в пространстве малой размерности, в худшем случае **O(n)**.

Здесь `n` — число узлов, `h` — высота дерева. При последовательном добавлении точек
дерево не обязательно остаётся сбалансированным.

### Структура узла дерева
Для заполнения дерева мы будем использовать объекты типа `RRTNode`. Каждый такой объект представляет собой узел дерева поиска, который хранит в себе следующую информацию:
* Координаты точки в пространстве (поле `state`)
* Ссылку на родительский узел (поле `parent`)
* Длину пути от корневого узла до текущего (поле `g`)
* Список непосредственных детей (поле `children`)

Конструктор создаёт пустой список `children` у каждой вершины. Если задан `parent`,
новая вершина сразу добавляется в `parent.children`.

### Пример работы с узлами и k-d-деревом
Для начала создадим корень дерева `RRTNode`, представляющий начальное состояние:

In [ ]:
root_node = RRTNode(state=(0, 0), parent=None, g=0)  # Создание корня дерева
tree = kdtree.create([root_node])  # Создание дерева KDTree

Затем добавим в дерево несколько новых узлов:

In [ ]:
cords = [(0, 1), (1, 2), (3, 5), (4, 2)]  # Набор координат новых узлов
for cord in cords:
    tree.add(RRTNode(state=cord, parent=root_node, g=math.hypot(*cord)))

Теперь можно найти ближайший узел к произвольной точке:

In [ ]:
state_query = (2, 3)  # Состояние, для которого ищем ближайший узел
nearest_node = tree.search_nn(state_query)[0].data  # Поиск ближайшего узла
print("Ближайший узел:", nearest_node.state)

## Вспомогательные функции

Препятствия задаются списками вершин многоугольников. Касание их границы также считается
столкновением. Функции `in_collision` и `is_trajectory_clear` уже реализованы.

**Задание.** Реализуйте `distance`, `in_goal_region`, `create_random_state`,
`find_nearest_neighbour` и `extend` согласно описаниям. Функция `extend` только создаёт
новую вершину; добавлять её в KD-дерево будет основной алгоритм.

In [ ]:
def in_collision(state: State, obstacles: Obstacles) -> bool:
    """Проверяет, лежит ли точка внутри препятствия или на его границе."""
    return any(ShapelyPolygon(obs).covers(Point(state)) for obs in obstacles)


def is_trajectory_clear(start_state: State, end_state: State,
                        obstacles: Obstacles) -> bool:
    """Проверяет, свободен ли от препятствий весь отрезок, включая концы."""
    if start_state == end_state:
        return not in_collision(start_state, obstacles)
    trajectory = LineString([start_state, end_state])
    return not any(trajectory.intersects(ShapelyPolygon(obs)) for obs in obstacles)


def validate_problem(map_w, map_h, obstacles, start, goal, max_transition,
                     max_iter, goal_region, goal_bias):
    """Проверяет параметры задачи; используется в обоих алгоритмах."""
    if map_w <= 0 or map_h <= 0 or max_transition <= 0:
        raise ValueError("Размеры карты и max_transition должны быть положительными")
    if max_iter < 0 or goal_region < 0 or not 0 <= goal_bias <= 1:
        raise ValueError("Нужны max_iter >= 0, goal_region >= 0 и 0 <= goal_bias <= 1")
    for state in (start, goal):
        if not (0 <= state[0] <= map_w and 0 <= state[1] <= map_h):
            raise ValueError("Старт и цель должны находиться в пределах карты")
        if in_collision(state, obstacles):
            raise ValueError("Старт и цель должны находиться вне препятствий")

In [ ]:
def distance(state1: State, state2: State) -> float:
    """
    Вычисляет евклидово расстояние между двумя состояниями (x, y).

    Args:
        state1: Координаты первого состояния.
        state2: Координаты второго состояния.

    Returns:
        float: Расстояние между состояниями.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


def in_goal_region(state: State, state_goal: State, region_size: float) -> bool:
    """
    Проверяет принадлежность состояния замкнутому кругу вокруг цели.

    Args:
        state: Координаты проверяемого состояния.
        state_goal: Координаты цели.
        region_size: Радиус целевой области.

    Returns:
        bool: True, если расстояние до цели не превышает region_size.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


def create_random_state(map_width: float, map_height: float,
                        goal_bias: float, goal_state: State,
                        goal_sampling_region: float) -> State:
    """
    Генерирует случайное состояние в пределах карты.

    С вероятностью goal_bias выбирает точку равномерно в круге вокруг цели,
    пересечённом с картой, иначе — равномерно на всей карте.
    Препятствия проверяются позже, при попытке расширить дерево.

    Args:
        map_width: Ширина карты.
        map_height: Высота карты.
        goal_bias: Вероятность выборки вблизи цели.
        goal_state: Координаты цели в пределах карты.
        goal_sampling_region: Радиус круга; при нуле выбирается сама цель.

    Returns:
        State: Случайная точка (x, y).

    Hint:
        Для равномерной выборки в круге используйте угол из [0, 2*pi]
        и радиус R * sqrt(random.random()). Точки вне карты отбрасывайте.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


def find_nearest_neighbour(tree: kdtree.KDNode, state_random: State) -> RRTNode:
    """
    Находит ближайшую к state_random вершину непустого KD-дерева.

    Hint:
        search_nn возвращает пару (узел KD-дерева, квадрат расстояния).
        Объект RRTNode хранится в поле data узла KD-дерева.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


def extend(tree: kdtree.KDNode, obstacles: Obstacles, state_random: State,
           max_transition: float) -> Tuple[bool, Optional[RRTNode]]:
    """
    Создаёт вершину в направлении state_random от ближайшей вершины дерева.

    Args:
        tree: Непустое KD-дерево поиска.
        obstacles: Список препятствий.
        state_random: Случайное состояние.
        max_transition: Максимальная длина нового ребра, больше нуля.

    Returns:
        (True, new_node), если отрезок свободен, иначе (False, None).
        При совпадении state_random с ближайшей вершиной расширение не выполняется.
        Новая вершина имеет parent и g и уже включена в parent.children,
        но ещё не добавлена в KD-дерево.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


### Промежуточные проверки вспомогательных функций

Все assert-ы собраны в одной функции `review_checks` в файле `review_checks.py`.
После реализации запускайте проверку соответствующего этапа. Здесь проверяются расстояния,
целевая область, сэмплирование, ближайший сосед, длина шага и столкновения.
Проверки сохраняют состояние `random`, поэтому не меняют последующие эксперименты.

In [ ]:
review_checks(
    "helpers", distance=distance, in_goal_region=in_goal_region,
    create_random_state=create_random_state, find_nearest_neighbour=find_nearest_neighbour,
    extend=extend, is_trajectory_clear=is_trajectory_clear, in_collision=in_collision,
)

## Основной алгоритм RRT

Функция `rrt` собирает вместе весь функционал алгоритма поиска пути. На каждом шаге:

1. Сгенерируйте случайное состояние с помощью `create_random_state`.
2. Попробуйте расширить дерево с помощью `extend`.
3. При успехе добавьте вершину в KD-дерево и в `all_points`.
4. Если новая вершина попала в целевую область, верните найденный путь.

**Задание.** Реализуйте `rrt`. В `all_points` храните пары `(RRTNode, номер итерации)`
в порядке добавления вершин. Нумерация итераций начинается с 1, корень имеет номер 0.
Проверьте отдельно случай, когда старт уже находится в целевой области.
При исчерпании бюджета верните неуспех вместе с построенным деревом.

Длина пути считается до вершины в целевой области; отдельное ребро до центра цели не добавляется.

In [ ]:
def rrt(map_w: float, map_h: float, obstacles: Obstacles,
        start_x: float, start_y: float, goal_x: float, goal_y: float,
        max_transition: float, max_iter: int = 3000,
        goal_region: float = 5, goal_bias: float = 0.05) -> SearchResult:
    """
    Реализует RRT с остановкой на первом найденном пути.

    Args:
        map_w, map_h: Размеры карты.
        obstacles: Список препятствий на карте.
        start_x, start_y: Координаты старта.
        goal_x, goal_y: Координаты цели.
        max_transition: Максимальная длина ребра.
        max_iter: Бюджет итераций, включая неудачные расширения.
        goal_region: Радиус целевой области и области выборки около цели.
        goal_bias: Вероятность выборки около цели.

    Returns:
        Пятёрку (found, end_node, iterations, tree, all_points).
        end_node равен None при неуспехе; дерево возвращается в любом случае.
        all_points содержит пары (вершина, итерация её добавления).
    """
    start, goal = (start_x, start_y), (goal_x, goal_y)
    validate_problem(map_w, map_h, obstacles, start, goal, max_transition,
                        max_iter, goal_region, goal_bias)
    node_start = RRTNode(start, None, 0.0)
    tree = kdtree.create([node_start])
    all_points = [(node_start, 0)]
    
    # <YOUR CODE GOES HERE>
    
    return False, None, max_iter, tree, all_points


### Промежуточные проверки RRT

Проверим старт в целевой области, нулевой бюджет, остановку на первом пути,
соблюдение длины шага и карту с непроходимой стеной.

In [ ]:
review_checks("rrt", planner=rrt)

# Тесты RRT

<img src="./media/maps_rrt.png" alt="Результаты RRT на двух картах" width="700"/>

## Небольшой тест

Предварительный тест на малой карте. Перед визуализацией `review_checks` проверяет
KD-дерево, отсутствие циклов и столкновений, ограничение `max_transition`, стоимости `g`
и восстановленный путь. Фиксируем seed для воспроизводимости.
Неуспех при ограниченном бюджете допустим; визуализация обрабатывает и этот случай.

In [ ]:
random.seed(42)
width, height, start, goal, obstacles = read_test_map("./tests/columns_small.txt")
max_transition, goal_region, goal_bias, max_iter = 5, 5, 0.05, 3000
rrt_result = rrt(
    width, height, obstacles, *start, *goal,
    max_transition=max_transition, max_iter=max_iter,
    goal_bias=goal_bias, goal_region=goal_region,
)
review_checks(
    "result", result=rrt_result, map_size=(width, height), start=start, goal=goal,
    obstacles=obstacles, max_transition=max_transition, max_iter=max_iter,
    goal_region=goal_region, algorithm="rrt",
)
found, end_node, number_of_steps, tree, all_points = rrt_result
print("Путь найден:", found, "Итераций:", number_of_steps)
print("Длина пути:", round(end_node.g, 2) if found else None)

In [ ]:
animation = visualize_rrt(
    width, height, *start, *goal, obstacles, all_points,
    construct_path_from_node(end_node), goal_region=goal_region,
    max_transition=max_transition, goal_bias=goal_bias,
    iterations=number_of_steps, max_iteration=max_iter,
    path_length=end_node.g if found else 0,
)
animation

## Колонны

In [ ]:
random.seed(42)
width, height, start, goal, obstacles = read_test_map("./tests/columns.txt")
max_transition, goal_region, goal_bias, max_iter = 7, 10, 0.05, 3000
rrt_result = rrt(
    width, height, obstacles, *start, *goal,
    max_transition=max_transition, max_iter=max_iter,
    goal_bias=goal_bias, goal_region=goal_region,
)
review_checks(
    "result", result=rrt_result, map_size=(width, height), start=start, goal=goal,
    obstacles=obstacles, max_transition=max_transition, max_iter=max_iter,
    goal_region=goal_region, algorithm="rrt",
)
found, end_node, number_of_steps, tree, all_points = rrt_result
print("Путь найден:", found, "Итераций:", number_of_steps)
print("Длина пути:", round(end_node.g, 2) if found else None)

In [ ]:
animation = visualize_rrt(
    width, height, *start, *goal, obstacles, all_points,
    construct_path_from_node(end_node), goal_region=goal_region,
    max_transition=max_transition, goal_bias=goal_bias,
    iterations=number_of_steps, max_iteration=max_iter,
    path_length=end_node.g if found else 0,
)
animation

## Случайные препятствия

In [ ]:
random.seed(42)
width, height, start, goal, obstacles = read_test_map("./tests/rand_polygons.txt")
max_transition, goal_region, goal_bias, max_iter = 7, 10, 0.05, 3000
rrt_result = rrt(
    width, height, obstacles, *start, *goal,
    max_transition=max_transition, max_iter=max_iter,
    goal_bias=goal_bias, goal_region=goal_region,
)
review_checks(
    "result", result=rrt_result, map_size=(width, height), start=start, goal=goal,
    obstacles=obstacles, max_transition=max_transition, max_iter=max_iter,
    goal_region=goal_region, algorithm="rrt",
)
found, end_node, number_of_steps, tree, all_points = rrt_result
print("Путь найден:", found, "Итераций:", number_of_steps)
print("Длина пути:", round(end_node.g, 2) if found else None)

In [ ]:
animation = visualize_rrt(
    width, height, *start, *goal, obstacles, all_points,
    construct_path_from_node(end_node), goal_region=goal_region,
    max_transition=max_transition, goal_bias=goal_bias,
    iterations=number_of_steps, max_iteration=max_iter,
    path_length=end_node.g if found else 0,
)
animation

## Массовые тесты

Переберём те же 27 комбинаций `max_transition`, `goal_region` и `goal_bias`
на малой карте. Для каждой комбинации используем пять seed и бюджет 1000 итераций.
Сетка параметров, seed и бюджет одинаковы для RRT и RRT*.

Каждый результат RRT проходит `review_checks`: проверяются все рёбра и стоимости,
а не только найденный путь. В таблице показываем долю успехов, среднее число итераций
и вершин, а также среднюю длину пути по успешным запускам.

In [ ]:
max_transitions = [3, 7, 15]
goal_regions = [5, 10, 25]
goal_biases = [0.05, 0.10, 0.15]
num_runs, test_budget = 5, 1000

width, height, start, goal, obstacles = read_test_map("./tests/columns_small.txt")
rrt_results = []
for max_transition in max_transitions:
    for goal_region in goal_regions:
        for goal_bias in goal_biases:
            run_results = []
            for seed in range(num_runs):
                random.seed(seed)
                result = rrt(
                    width, height, obstacles, *start, *goal,
                    max_transition=max_transition, max_iter=test_budget,
                    goal_bias=goal_bias, goal_region=goal_region,
                )
                review_checks(
                    "result", result=result, map_size=(width, height), start=start,
                    goal=goal, obstacles=obstacles, max_transition=max_transition,
                    max_iter=test_budget, goal_region=goal_region, algorithm="rrt",
                )
                found, end_node, steps, tree, all_points = result
                run_results.append((found, steps, len(all_points), end_node.g if found else None))
            lengths = [run[3] for run in run_results if run[0]]
            rrt_results.append({
                "max_transition": max_transition,
                "goal_region": goal_region,
                "goal_bias": goal_bias,
                "max_iter": test_budget,
                "success_rate": mean(run[0] for run in run_results),
                "avg_steps": mean(run[1] for run in run_results),
                "avg_nodes": mean(run[2] for run in run_results),
                "avg_path_length": mean(lengths) if lengths else None,
            })
rrt_df = pd.DataFrame(rrt_results)
rrt_df

# Алгоритм RRT*

RRT быстро находит допустимый путь, но родитель каждой вершины после добавления не меняется.
RRT* использует новые вершины, чтобы улучшать уже построенные пути. Основные отличия:

1. **Выбор родителя.** Для новой точки рассмотрите соседей и выберите допустимое ребро,
   минимизирующее `parent.g + distance(parent.state, new_node.state)`.
2. **Смена родителя.** Проверьте, можно ли уменьшить стоимость пути к соседу,
   сделав новую вершину его родителем. После смены родителя обновите `g` всех потомков.
3. **Продолжение поиска.** Используйте весь бюджет итераций и в конце выберите вершину
   с наименьшим `g` среди всех вершин целевой области.

RRT* обладает асимптотической оптимальностью при выполнении условий на пространство,
выборку и радиус соседства. За конечное число итераций оптимальность не гарантируется.
Описание алгоритма: [S. Karaman, E. Frazzoli, 2010](https://roboticsproceedings.org/rss06/p34.pdf).

## Поиск соседей и выбор родителя

Для двумерного пространства используем радиус

$$r_n = \min\left(\mathrm{max\_transition},\ \gamma\sqrt{\frac{\log n}{n}}\right),$$

где `n` — число вершин с учётом добавляемой, а параметр `rewire_radius` задаёт коэффициент
$\gamma$, а не постоянный радиус. По умолчанию берём $2\sqrt{\mathrm{map\_w}\,\mathrm{map\_h}}$,
используя площадь карты как верхнюю оценку площади свободного пространства.

**Задание.** Реализуйте `find_near_neighbours` и `choose_parent`.

Функция `find_near_neighbours` должна вернуть все вершины на расстоянии меньше
`radius` от новой точки. Точки ровно на границе не включаем. Для поиска используйте
`tree.search_nn_dist`: KD-дерево позволяет пропускать ветви, удалённые от новой точки.

В `kdtree 0.16` есть ошибка: при проверке точек метод использует квадрат расстояния,
а при выборе ветвей для поиска — обычное расстояние вдоль координатной оси.
При этом в обоих случаях он сравнивает расстояние с одним и тем же порогом.
Например, для радиуса `0.5` квадрат равен `0.25`. Если передать `0.25`, библиотека
может отбросить ветвь с нужными точками: для выбора ветвей этот порог слишком мал.

Поэтому вызывайте `tree.search_nn_dist(state, max(radius, radius ** 2))`.
Так запрос не пропустит нужные ветви, но может вернуть лишние вершины.
Из полученного списка оставьте только те, для которых
`distance(node.state, state) < radius`.

Функция `choose_parent` выбирает, через какую вершину короче добраться от старта
до новой точки. Для каждого соседа посчитайте длину такого пути:
`node.g + distance(node.state, new_node.state)`. Если она меньше текущего `new_node.g`
и отрезок от соседа до новой точки не пересекает препятствия, замените `parent` и `g`
у новой вершины. При смене родителя удалите вершину из `children` старого родителя
и добавьте её в `children` нового.

Изначально у новой вершины уже есть родитель, выбранный функцией `extend`.
Если ни один сосед не даёт более короткого пути, оставьте этого родителя.
Он может находиться за пределами круга поиска соседей: радиус этого круга уменьшается
при росте дерева. Это допустимо, поскольку `extend` уже проверила ребро и ограничила
его длину значением `max_transition`.

Максимальная длина ребра остаётся такой же, как у RRT: `max_transition`.
Радиус поиска соседей также не превышает это значение.


In [ ]:
def find_near_neighbours(tree: kdtree.KDNode, state: State,
                         radius: float) -> List[RRTNode]:
    """Возвращает вершины на расстоянии меньше radius от state."""
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


def choose_parent(new_node: RRTNode, near_nodes: List[RRTNode],
                  obstacles: Obstacles) -> RRTNode:
    """
    Выбирает родителя с минимальной стоимостью пути к новой вершине.

    Args:
        new_node: Вершина из extend с допустимым исходным родителем и стоимостью g.
        near_nodes: Вершины окрестности новой точки.
        obstacles: Препятствия, которые нельзя пересекать.

    Returns:
        Ту же вершину с обновлёнными parent и g и согласованными списками children.
        В KD-дерево её пока не добавляет.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


## Улучшение путей к соседним вершинам

**Задание.** Реализуйте `rewire`. Если путь через `new_node` короче и ребро свободно,
измените родителя соседней вершины:

1. Удалите вершину из `children` старого родителя.
2. Измените её `parent` и добавьте её в `new_node.children`.
3. Пересчитайте её `g`, затем обойдите её потомков через списки `children` и обновите их `g`.

Для каждой вершины должно выполняться
`node.g = node.parent.g + distance(node.parent.state, node.state)`.
Не меняйте родителя у корня, новой вершины и её предков: дерево должно оставаться без циклов.
Координаты вершин не меняются, поэтому перестраивать KD-дерево не требуется.


In [ ]:
def rewire(new_node: RRTNode, near_nodes: List[RRTNode],
           obstacles: Obstacles) -> None:
    """
    Улучшает пути к соседям через new_node и обновляет g их потомков.

    При смене родителя обновляет parent и списки children у старого и нового родителя.
    Стоимости потомков пересчитывает обходом children. Изменяет дерево на месте.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


### Промежуточные проверки операций RRT*

Проверим поиск соседей, включая границу и малые радиусы, выбор более выгодного родителя,
смену родителей у нескольких соседних вершин и обновление стоимости детей и внуков.
Также проверяются списки `children`: после смены родителя вершина должна быть
только в списке нового родителя, без повторов. Дополнительно проверяется запрет
рёбер сквозь препятствия и сохранение дерева без циклов.

In [ ]:
review_checks(
    "rrtstar_helpers", find_near_neighbours=find_near_neighbours,
    choose_parent=choose_parent, rewire=rewire,
)

## Основной алгоритм RRT*

**Задание.** Реализуйте `rrtstar` с тем же форматом результата, что и у `rrt`.
После успешного `extend` найдите соседей, выберите родителя, добавьте вершину в дерево
и попробуйте сократить пути к соседним вершинам через новую вершину. Даже после первого попадания в целевую область продолжайте поиск.

При выборе результата используйте актуальные значения `g`: стоимость уже найденной
целевой вершины могла уменьшиться после смены родителя у одного из её предков.
Если старт уже находится в целевой области, путь нулевой длины оптимален и можно сразу его вернуть.

In [ ]:
def rrtstar(map_w: float, map_h: float, obstacles: Obstacles,
            start_x: float, start_y: float, goal_x: float, goal_y: float,
            max_transition: float, max_iter: int = 3000,
            goal_region: float = 5, goal_bias: float = 0.05,
            rewire_radius: Optional[float] = None) -> SearchResult:
    """
    Реализует RRT* с выбором и сменой родителей для сокращения путей.

    Параметры и возвращаемая пятёрка совпадают с rrt.
    rewire_radius задаёт положительный коэффициент gamma в формуле r_n.
    При None используется 2 * sqrt(map_w * map_h).
    Кроме случая старта в целевой области, выполняет ровно max_iter итераций.
    """
    start, goal = (start_x, start_y), (goal_x, goal_y)
    validate_problem(map_w, map_h, obstacles, start, goal, max_transition,
                            max_iter, goal_region, goal_bias)
    gamma = 2 * math.sqrt(map_w * map_h) if rewire_radius is None else rewire_radius
    if gamma <= 0:
            raise ValueError("rewire_radius должен быть положительным")
    node_start = RRTNode(start, None, 0.0)
    tree = kdtree.create([node_start])
    all_points = [(node_start, 0)]
    
    # <YOUR CODE GOES HERE>
    
    return False, None, max_iter, tree, all_points


### Промежуточные проверки RRT*

Используем те же пограничные случаи, что и для RRT. Дополнительно проверяем,
что RRT* продолжает поиск после первого решения, возвращает лучшую целевую вершину
и при увеличении бюджета на том же потоке выборки не ухудшает лучший путь.

In [ ]:
review_checks("rrtstar", planner=rrtstar)

# Тесты RRT*

<img src="./media/maps_rrtstar.png" alt="Результаты RRT* на двух картах" width="700"/>

### Визуализация итогового дерева

Ниже используются те же три карты, параметры и seed, что и в примерах RRT:
«Небольшой тест», «Колонны» и «Случайные препятствия». Перед каждой визуализацией проверяется всё дерево.

Родителем вершины может стать другая вершина, добавленная позже.
Поэтому порядок появления вершин не описывает историю изменения родителей. Для RRT* рисуем окончательное дерево и лучший путь
теми же цветами, что и в примерах RRT.

Эти рисунки показывают результат после завершения поиска. Появление первого пути и его улучшение в процессе поиска здесь не анимируются.

In [ ]:
def show_search_tree(map_w, map_h, obstacles, start, goal, result,
                     goal_region, title="RRT*", max_transition=None,
                     goal_bias=None, max_iteration=None):
    """Рисует итоговое дерево и путь после завершения поиска."""
    found, end_node, iterations, tree, points = result
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set(xlim=(0, map_w), ylim=(0, map_h), aspect="equal")
    ax.set_facecolor("#e9fce1")
    for obstacle in obstacles:
        ax.add_patch(MPolygon(obstacle, closed=True, edgecolor="black",
                              facecolor="#f5c19c", alpha=0.7))
    ax.add_patch(Circle(goal, goal_region, color="red", alpha=0.3))
    segments = [[node.parent.state, node.state] for node, _ in points
                if node.parent is not None]
    ax.add_collection(LineCollection(segments, colors="skyblue", alpha=0.6))
    ax.scatter([node.state[0] for node, _ in points],
               [node.state[1] for node, _ in points],
               c=range(len(points)), cmap="viridis", s=8)
    path = construct_path_from_node(end_node)
    if found:
        ax.plot([node.state[0] for node in path], [node.state[1] for node in path],
                color="orange", linewidth=3, label="Путь")
    ax.plot(*start, "go", markersize=10, label="Старт")
    ax.plot(*goal, "ro", markersize=10, label="Цель")
    length = f"{end_node.g:.2f}" if found else "не найден"
    ax.set_title(
        f"{title}: итераций {iterations}/{max_iteration}, длина пути {length}\n"
        f"Макс. ребро: {max_transition}; выборка у цели: {goal_bias}; радиус цели: {goal_region}"
    )
    ax.legend()
    plt.show()

## Небольшой тест

Запустим RRT* с теми же параметрами, что и RRT. Проверка результата дополнительно
контролирует полный расход бюджета и минимальную стоимость целевой вершины.

In [ ]:
random.seed(42)
width, height, start, goal, obstacles = read_test_map("./tests/columns_small.txt")
max_transition, goal_region, goal_bias, max_iter = 5, 5, 0.05, 3000
star_result = rrtstar(
    width, height, obstacles, *start, *goal,
    max_transition=max_transition, max_iter=max_iter,
    goal_bias=goal_bias, goal_region=goal_region,
)
review_checks(
    "result", result=star_result, map_size=(width, height), start=start, goal=goal,
    obstacles=obstacles, max_transition=max_transition, max_iter=max_iter,
    goal_region=goal_region, algorithm="rrtstar",
)
found, end_node, number_of_steps, tree, all_points = star_result
print("Путь найден:", found, "Итераций:", number_of_steps)
print("Длина пути:", round(end_node.g, 2) if found else None)

In [ ]:
show_search_tree(
    width, height, obstacles, start, goal, star_result, goal_region,
    max_transition=max_transition, goal_bias=goal_bias, max_iteration=max_iter,
)

## Колонны

In [ ]:
random.seed(42)
width, height, start, goal, obstacles = read_test_map("./tests/columns.txt")
max_transition, goal_region, goal_bias, max_iter = 7, 10, 0.05, 3000
star_result = rrtstar(
    width, height, obstacles, *start, *goal,
    max_transition=max_transition, max_iter=max_iter,
    goal_bias=goal_bias, goal_region=goal_region,
)
review_checks(
    "result", result=star_result, map_size=(width, height), start=start, goal=goal,
    obstacles=obstacles, max_transition=max_transition, max_iter=max_iter,
    goal_region=goal_region, algorithm="rrtstar",
)
found, end_node, number_of_steps, tree, all_points = star_result
print("Путь найден:", found, "Итераций:", number_of_steps)
print("Длина пути:", round(end_node.g, 2) if found else None)

In [ ]:
show_search_tree(
    width, height, obstacles, start, goal, star_result, goal_region,
    max_transition=max_transition, goal_bias=goal_bias, max_iteration=max_iter,
)

## Случайные препятствия

In [ ]:
random.seed(42)
width, height, start, goal, obstacles = read_test_map("./tests/rand_polygons.txt")
max_transition, goal_region, goal_bias, max_iter = 7, 10, 0.05, 3000
star_result = rrtstar(
    width, height, obstacles, *start, *goal,
    max_transition=max_transition, max_iter=max_iter,
    goal_bias=goal_bias, goal_region=goal_region,
)
review_checks(
    "result", result=star_result, map_size=(width, height), start=start, goal=goal,
    obstacles=obstacles, max_transition=max_transition, max_iter=max_iter,
    goal_region=goal_region, algorithm="rrtstar",
)
found, end_node, number_of_steps, tree, all_points = star_result
print("Путь найден:", found, "Итераций:", number_of_steps)
print("Длина пути:", round(end_node.g, 2) if found else None)

In [ ]:
show_search_tree(
    width, height, obstacles, start, goal, star_result, goal_region,
    max_transition=max_transition, goal_bias=goal_bias, max_iteration=max_iter,
)

## Массовые тесты

Переберём те же 27 комбинаций `max_transition`, `goal_region` и `goal_bias`
на малой карте. Для каждой комбинации используем пять seed и бюджет 1000 итераций.
Сетка параметров, seed и бюджет одинаковы для RRT и RRT*.

Каждый результат RRT* проходит `review_checks`: проверяются все рёбра и стоимости,
а не только найденный путь. В таблице показываем долю успехов, среднее число итераций
и вершин, а также среднюю длину пути по успешным запускам.

In [ ]:
max_transitions = [3, 7, 15]
goal_regions = [5, 10, 25]
goal_biases = [0.05, 0.10, 0.15]
num_runs, test_budget = 5, 1000

width, height, start, goal, obstacles = read_test_map("./tests/columns_small.txt")
rrtstar_results = []
for max_transition in max_transitions:
    for goal_region in goal_regions:
        for goal_bias in goal_biases:
            run_results = []
            for seed in range(num_runs):
                random.seed(seed)
                result = rrtstar(
                    width, height, obstacles, *start, *goal,
                    max_transition=max_transition, max_iter=test_budget,
                    goal_bias=goal_bias, goal_region=goal_region,
                )
                review_checks(
                    "result", result=result, map_size=(width, height), start=start,
                    goal=goal, obstacles=obstacles, max_transition=max_transition,
                    max_iter=test_budget, goal_region=goal_region, algorithm="rrtstar",
                )
                found, end_node, steps, tree, all_points = result
                run_results.append((found, steps, len(all_points), end_node.g if found else None))
            lengths = [run[3] for run in run_results if run[0]]
            rrtstar_results.append({
                "max_transition": max_transition,
                "goal_region": goal_region,
                "goal_bias": goal_bias,
                "max_iter": test_budget,
                "success_rate": mean(run[0] for run in run_results),
                "avg_steps": mean(run[1] for run in run_results),
                "avg_nodes": mean(run[2] for run in run_results),
                "avg_path_length": mean(lengths) if lengths else None,
            })
rrtstar_df = pd.DataFrame(rrtstar_results)
rrtstar_df

## Сравнение RRT и RRT*

Для каждого алгоритма и бюджета используем одни и те же seed и параметры карты.
RRT останавливается на первом пути, а RRT* расходует весь бюджет на его улучшение.
Поэтому `avg_steps` может различаться даже при одинаковом `max_iter`.

Среднюю длину считаем только по успешным запускам и показываем вместе с долей успехов.
Один запуск не обязан дать RRT* преимущество: сравните несколько seed и бюджетов.
Время работы здесь не измеряется; поиск более выгодных родителей требует дополнительных вычислений.

In [ ]:
width, height, start, goal, obstacles = read_test_map("./tests/columns_small.txt")
comparison = []
for budget in [500, 1500]:
    for name, planner in [("RRT", rrt), ("RRT*", rrtstar)]:
        runs = []
        for seed in range(5):
            random.seed(seed)
            result = planner(
                width, height, obstacles, *start, *goal,
                max_transition=5, max_iter=budget, goal_region=5, goal_bias=0.05,
            )
            review_checks(
                "result", result=result, map_size=(width, height), start=start,
                goal=goal, obstacles=obstacles, max_transition=5, max_iter=budget,
                goal_region=5, algorithm="rrt" if name == "RRT" else "rrtstar",
            )
            found, end_node, steps, tree, points = result
            runs.append((found, end_node.g if found else None, steps, len(points)))
        lengths = [run[1] for run in runs if run[0]]
        comparison.append({
            "algorithm": name,
            "max_iter": budget,
            "success_rate": mean(run[0] for run in runs),
            "avg_path_length": mean(lengths) if lengths else None,
            "avg_steps": mean(run[2] for run in runs),
            "avg_nodes": mean(run[3] for run in runs),
        })
comparison_df = pd.DataFrame(comparison)
comparison_df

**Вопросы для анализа.** Как меняется длина пути RRT* при увеличении бюджета?
Почему важно обновлять `g` потомков? Что произойдёт, если забыть проверить ребро
к новому родителю на столкновения? Как коэффициент `rewire_radius` влияет
на качество пути и время работы? Повторите сравнение на картах «Колонны» и «Случайные препятствия».